# MoE-MedIR — Kaggle Fine-Tune Notebook

> **Paper**: Bridging the Intra-Modal Heterogeneity Gap: Mixture-of-Experts for Multi-Modality Medical Image Retrieval
> **Conference**: ICARCV 2026

Fine-tune backbone + MoE head end-to-end trên raw images (không dùng pre-extracted features).

**Chiến lược 2 Stage:**
- Stage 1 (`frozen_epochs` đầu): backbone frozen, chỉ train MoE head (warm-up)
- Stage 2 (còn lại): unfreeze last N blocks (ViT) / full model (CNN), backbone_lr nhỏ hơn

**CNN** (convnext_base): unfreeze toàn bộ model ở Stage 2
**ViT** (clip, biomedclip, dino): chỉ unfreeze `finetune_blocks` blocks cuối + final norm

**Setup**: Notebook → Settings → Accelerator → GPU T4/P100
**Internet**: Settings → Internet → **On**

## Cell 1 — Check GPU

In [1]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Settings → Accelerator → GPU')

CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2 — Install Packages

In [2]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'open_clip_torch', 'timm', 'transformers',
    'pytorch-metric-learning', 'scikit-learn',
    'matplotlib', 'seaborn'], check=True)
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## Cell 3 — Clone Project từ GitHub

**Kaggle**: Settings → Internet phải **On**.

In [3]:
import os, subprocess

REPO_URL     = 'https://github.com/trong5nhan6/moe_medir.git'
PROJECT_ROOT = '/kaggle/working/moe_medir'

if os.path.exists(PROJECT_ROOT):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'])

Cloning into '/kaggle/working/moe_medir'...


Working directory: /kaggle/working/moe_medir
31df711 set finetune_blocks default to 2 for gem and dolg baselines
28e75ec set kaggle_finetune default backbone to convnext_base
e43e0ce fix CNN finetune_blocks: respect n_blocks in unfreeze_partial


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0)

## Cell 4 — ⚙️ Config Editor

**Chỉnh sửa tại đây** trước khi chạy fine-tune.

| Tham số | Các lựa chọn |
|---|---|
| `BACKBONE` | `clip_vitb32` · `biomedclip` · `dinov2_vitb14` · `convnext_base` |
| `FEATURE_MODE` | `concat` (CLS+Patch / GAP+GMP, 2×dim) · `cls` (CLS / GAP only, dim) |
| `ROUTING_MODE` | `token_choice` · `expert_choice` |
| `TOP_K` | 2 (sparse) · 4 (dense) |

**Fine-tuning hyperparams:**

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `FROZEN_EPOCHS` | 5 | Stage 1: warm-up với backbone frozen |
| `FINETUNE_BLOCKS` | 2 | Số blocks cuối unfreeze (ViT) |
| `BACKBONE_LR` | 1e-5 | LR cho backbone (Stage 2) |
| `HEAD_LR` | 1e-4 | LR cho MoE head (cả 2 stage) |

**Loss config** — `Total = SupCon + λ_lb·LB + λ_orth·Orth + λ_aff·Affinity`:

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `TEMPERATURE` | 0.07 | SupCon temperature τ (nhỏ → contrastive gắt hơn) |
| `LAMBDA_LB` | 0.01 | Load-balance loss (chỉ `token_choice`; `expert_choice` không cần) |
| `LAMBDA_ORTH` | 0.01 | Expert orthogonality — đẩy experts khác biệt nhau |
| `LAMBDA_AFFINITY` | 0.1 | Modality affinity — routing đa dạng theo modality |
| `FEAT_NOISE` | 0.01 | Gaussian noise std trên input features (regularization) |

In [ ]:
import re, sys, importlib

# ════════════════════════════════════════════════════════════════
#  ✏️  CHỈNH SỬA CONFIG TẠI ĐÂY
# ════════════════════════════════════════════════════════════════

BACKBONE      = "dinov2_vitb14"  # clip_vitb32 | biomedclip | dinov2_vitb14 | convnext_base
FEATURE_MODE  = "concat"         # concat | cls
ROUTING_MODE  = "token_choice"   # token_choice | expert_choice
TOP_K         = 2
EMBED_DIM     = 256
NUM_EXPERTS   = 8
BATCH_SIZE    = 256
EPOCHS        = 1

# Fine-tuning specific
FROZEN_EPOCHS   = 5     # Stage 1: frozen warm-up epochs
FINETUNE_BLOCKS = 2     # CNN: số stages cuối unfreeze (0=toàn bộ, dễ OOM); ViT: số blocks cuối
BACKBONE_LR     = 1e-5  # LR cho backbone (Stage 2)
HEAD_LR         = 1e-4  # LR cho MoE head

# ShareMoE (DeepSeek-MoE style)
USE_SHAREMOE       = True  # True: bật shared experts
NUM_SHARED_EXPERTS = 2      # số experts luôn active (chỉ dùng khi USE_SHAREMOE=True)
                            # ví dụ: NUM_EXPERTS=8, NUM_SHARED_EXPERTS=2 → 2 shared + 6 routed

# ── Loss config (multi-task objective) ──────────────────────────
# Tổng loss = SupCon + λ_lb·LB + λ_orth·Orth + λ_aff·Affinity
TEMPERATURE     = 0.07  # SupCon temperature τ (nhỏ → contrastive gắt hơn)
LAMBDA_LB       = 0.01  # load-balance loss weight (chỉ token_choice; expert_choice không cần)
LAMBDA_ORTH     = 0.01  # expert weight orthogonality loss weight (đa dạng experts)
LAMBDA_AFFINITY = 0.1   # modality affinity / routing diversity loss weight
FEAT_NOISE      = 0.01  # Gaussian noise std thêm vào input features (regularization)

# ════════════════════════════════════════════════════════════════

def _patch(content, field, value):
    if isinstance(value, str):
        pattern     = rf'([ \t]+{field}\s*:\s*\w+\s*=\s*)["\'].*?["\']'
        replacement = rf'\g<1>"{value}"'
    elif isinstance(value, bool):
        pattern     = rf'([ \t]+{field}\s*:\s*bool\s*=\s*)(?:True|False)'
        replacement = rf'\g<1>{value}'
    else:
        pattern     = rf'([ \t]+{field}\s*:\s*[\w\[\]]+\s*=\s*)[\d.e+\-]+'
        replacement = rf'\g<1>{value}'
    new, n = re.subn(pattern, replacement, content)
    if n == 0:
        print(f"  [warn] field '{field}' not found")
    return new

with open('config.py', 'r') as f:
    cfg_text = f.read()

for field, val in [
    ('backbone',           BACKBONE),
    ('feature_mode',       FEATURE_MODE),
    ('routing_mode',       ROUTING_MODE),
    ('top_k',              TOP_K),
    ('embed_dim',          EMBED_DIM),
    ('num_experts',        NUM_EXPERTS),
    ('batch_size',         BATCH_SIZE),
    ('epochs',             EPOCHS),
    ('lr',                 HEAD_LR),
    ('use_sharemoe',       USE_SHAREMOE),
    ('num_shared_experts', NUM_SHARED_EXPERTS),
    ('temperature',        TEMPERATURE),
    ('lambda_lb',          LAMBDA_LB),
    ('lambda_orth',        LAMBDA_ORTH),
    ('lambda_affinity',    LAMBDA_AFFINITY),
    ('feat_noise',         FEAT_NOISE),
]:
    cfg_text = _patch(cfg_text, field, val)

with open('config.py', 'w') as f:
    f.write(cfg_text)

if 'config' in sys.modules:
    import config as _c; importlib.reload(_c); from config import CFG
else:
    from config import CFG

print("=" * 60)
print("  Config hiện tại")
print("=" * 60)
print(f"  backbone      : {CFG.backbone}")
print(f"  feature_mode  : {CFG.feature_mode}  →  feature_dim={CFG.feature_dim}")
print(f"  routing_mode  : {CFG.routing_mode}")
print(f"  top_k         : {CFG.top_k}  /  num_experts={CFG.num_experts}")
print(f"  embed_dim     : {CFG.embed_dim}")
print(f"  batch_size    : {CFG.batch_size}  epochs={CFG.epochs}")
if CFG.use_sharemoe:
    n_routed = CFG.num_experts - CFG.num_shared_experts
    print(f"  ShareMoE      : ON  →  {CFG.num_shared_experts} shared + {n_routed} routed")
else:
    print(f"  ShareMoE      : OFF  (standard MoE)")
print("=" * 60)
print(f"  frozen_epochs  : {FROZEN_EPOCHS}  (Stage 1 warm-up)")
print(f"  finetune_blocks: {FINETUNE_BLOCKS}  ({'2 stages cuối CNN ~8M params' if FINETUNE_BLOCKS > 0 else 'toàn bộ backbone'})")
print(f"  backbone_lr    : {BACKBONE_LR}  (Stage 2)")
print(f"  head_lr        : {HEAD_LR}")
print("=" * 60)
print("  Loss weights")
print(f"  temperature τ  : {CFG.temperature}  (SupCon)")
_lb_note = "" if CFG.routing_mode == "token_choice" else "  (bỏ qua ở expert_choice)"
print(f"  λ_lb           : {CFG.lambda_lb}  (load-balance){_lb_note}")
print(f"  λ_orth         : {CFG.lambda_orth}  (orthogonality)")
print(f"  λ_affinity     : {CFG.lambda_affinity}  (modality affinity)")
print(f"  feat_noise     : {CFG.feat_noise}  (input noise std)")
print("=" * 60)
print("✅  config.py đã được cập nhật.")

## Cell 5 — Fine-Tune: 2-Stage Training

Chạy `train_finetune.py` với cấu hình từ Cell 4.

**Tiến trình:**
- **Stage 1** (epochs 1–`FROZEN_EPOCHS`): Backbone frozen → chỉ MoE head học
- **Stage 2** (epochs `FROZEN_EPOCHS+1`–`EPOCHS`): Unfreeze → cả backbone + head học

Log hiển thị `[S1]` / `[S2]` mỗi batch. Val mAP@R được đánh giá mỗi 5 epochs.

~25–40 phút trên P100.

In [5]:
import subprocess, sys
subprocess.run([
    sys.executable, 'train_finetune.py',
    '--model', 'moe',
    '--epochs',          str(EPOCHS),
    '--frozen_epochs',   str(FROZEN_EPOCHS),
    '--finetune_blocks', str(FINETUNE_BLOCKS),
    '--backbone_lr',     str(BACKBONE_LR),
    '--head_lr',         str(HEAD_LR),
], check=True)

Run             : finetune_dinov2_vitb14_moe
Device          : cuda  |  Seed: 42
Backbone        : dinov2_vitb14  |  feature_mode=concat  |  feature_dim=1536
Stage 1 frozen  : 5 epochs  →  head_lr=0.0001
Stage 2 unfreeze: last 2 blocks  →  backbone_lr=1e-05


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1711.38it/s, Materializing param=layernorm.weight]                                 


Head params     : 9,008,134 trainable
Backbone params : 86,580,480 (frozen until Stage 2)


100%|██████████| 206M/206M [01:29<00:00, 2.31MB/s] 
100%|██████████| 19.7M/19.7M [00:20<00:00, 978kB/s] 
100%|██████████| 54.9M/54.9M [00:07<00:00, 7.81MB/s]
100%|██████████| 35.5M/35.5M [00:35<00:00, 999kB/s] 


Train batches/epoch: 15


Ep   1 [S1] | loss=4.1262 (sc=4.115 lb=1.147 orth=0.000 aff=-0.004) | avg mAP@R=31.05 | path=38.8  derm=40.9  octm=23.5  bloo=21.0
         -> New best: 31.05  saved to results/checkpoints/best_finetune_dinov2_vitb14_moe.pt

Done. Best val mAP@R: 31.05
History : results/history_finetune_dinov2_vitb14_moe.csv
Config  : results/config_finetune_dinov2_vitb14_moe.json


CompletedProcess(args=['/usr/bin/python3', 'train_finetune.py', '--model', 'moe', '--epochs', '1', '--frozen_epochs', '5', '--finetune_blocks', '2', '--backbone_lr', '1e-05', '--head_lr', '0.0001'], returncode=0)

## Cell 6 — Evaluate Fine-tuned Model

In [6]:
import subprocess, sys
from config import CFG
run_name = f'finetune_{CFG.backbone}_moe'
subprocess.run([sys.executable, 'eval/evaluate_finetune.py',
                '--name', run_name], check=True)

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1682.84it/s, Materializing param=layernorm.weight]                                 


Loaded  : results/checkpoints/best_finetune_dinov2_vitb14_moe.pt
Backbone: dinov2_vitb14  |  epoch=1  |  best val mAP@R=31.05

Dataset           mAP@R     MRR     R@1     R@5    R@10   MPR@1   MPR@5  MPR@10
-------------------------------------------------------------------------------
pathmnist         55.40   95.40   92.86   98.58   99.36   92.86   91.35   90.16
dermamnist        40.23   74.70   64.94   87.08   93.32   64.94   63.31   62.66
octmnist          15.65   72.75   58.20   92.40   97.20   58.20   53.74   50.40
bloodmnist        20.27   78.16   67.52   92.43   96.49   67.52   63.69   60.93
-------------------------------------------------------------------------------
Average                   32.89

Saved: results/test_finetune_dinov2_vitb14_moe.csv


CompletedProcess(args=['/usr/bin/python3', 'eval/evaluate_finetune.py', '--name', 'finetune_dinov2_vitb14_moe'], returncode=0)

In [12]:
import subprocess, sys, os
import importlib, config as _cfg_mod
importlib.reload(_cfg_mod)
from config import CFG

run_name  = f'finetune_{CFG.backbone}_moe'
ckpt_path = os.path.join(CFG.checkpoint_dir, f'best_{run_name}.pt')

if not os.path.exists(ckpt_path):
    print(f'⏭️  Bỏ qua moe — chưa có checkpoint: {ckpt_path}')
else:
    print(f'\n>>> SuperGlobal re-ranking: moe finetune (test set)')
    subprocess.run([
        sys.executable, 'baselines/superglobal.py',
        '--model', 'moe',
        '--ckpt',  ckpt_path,
        '--alpha', '3.0',
        '--top_k', '10',
        '--split', 'test',
    ], check=True)


>>> SuperGlobal re-ranking: moe finetune (test set)


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1703.72it/s, Materializing param=layernorm.weight]                                 



SuperGlobal (α-QE) re-ranking
  model=moe  ckpt=best_finetune_dinov2_vitb14_moe.pt  alpha=3.0  top_k=10  split=test
  pathmnist     mAP@R: 55.40 → 60.75 (+5.35)  |  R@1: 92.9 → 93.0
  dermamnist    mAP@R: 40.23 → 39.71 (-0.52)  |  R@1: 64.9 → 64.3
  octmnist      mAP@R: 15.65 → 16.64 (+0.99)  |  R@1: 58.2 → 59.5
  bloodmnist    mAP@R: 20.27 → 22.10 (+1.83)  |  R@1: 67.5 → 68.2
──────────────────────────────────────────────────────────────────────
  avg           mAP@R: 32.89 → 34.80 (+1.91)

SuperGlobal re-ranking improved avg mAP@R by +1.91


## Cell 7 — Training History & Best Score

In [7]:
import pandas as pd, os
from config import CFG

run_name  = f'finetune_{CFG.backbone}_moe'
hist_path = f'results/history_{run_name}.csv'

if os.path.exists(hist_path):
    df = pd.read_csv(hist_path)
    print('=== Training History ===')
    cols = ['epoch', 'stage', 'avg_mAP@R', 'loss', 'sc_loss', 'orth_loss']
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print()
    best_row = df.loc[df['avg_mAP@R'].idxmax()]
    print(f"Best val mAP@R: {best_row['avg_mAP@R']:.2f}  (epoch {int(best_row['epoch'])}, Stage {int(best_row['stage'])})")
else:
    print(f'File chưa có: {hist_path}')

=== Training History ===
 epoch  stage  avg_mAP@R   loss  sc_loss  orth_loss
     1      1      31.05 4.1262   4.1152        0.0

Best val mAP@R: 31.05  (epoch 1, Stage 1)


## Cell 8 — Save Results

Tất cả file trong `/kaggle/working/` tự động lưu — tải về qua tab **Output**.

In [8]:
import os
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1024:.1f} KB)')

results/config_finetune_dinov2_vitb14_moe.json  (1.2 KB)
results/history_finetune_dinov2_vitb14_moe.csv  (0.8 KB)
results/test_finetune_dinov2_vitb14_moe.csv  (0.3 KB)
results/checkpoints/best_finetune_dinov2_vitb14_moe.pt  (373506.2 KB)
